In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, KBinsDiscretizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import ttest_rel, wilcoxon
import warnings
warnings.filterwarnings('ignore')

# Load data
adult = fetch_openml('adult', version=2, as_frame=True)
df = adult.frame
df['income'] = df['class'].map({'<=50K': 0, '>50K': 1}).astype(int)
df = df.replace(' ?', np.nan)

# Features and target
X = df.drop(['class', 'income'], axis=1)
y = df['income']

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

print(f"Train: {X_train.shape}, Dev: {X_dev.shape}, Test: {X_test.shape}")

Train: (35165, 14), Dev: (3908, 14), Test: (9769, 14)


## TASK 1: Create & Justify Engineered Features

In [2]:

def feature_engineering(df):
    """Add 6 engineered features to the dataframe."""
    df = df.copy()
    
    # 1. Age buckets (binned)
    df['age_bucket'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 100], 
                              labels=['<25', '25-34', '35-44', '45-54', '55+'])
    
    # 2. Hours-per-week bins
    df['hours_bucket'] = pd.cut(df['hours-per-week'], bins=[0, 20, 40, 60, 100],
                                labels=['part-time', 'full-time', 'overtime', 'heavy'])
    
    # 3. Flag: capital_gain > 0
    df['has_capital_gain'] = (df['capital-gain'] > 0).astype(int)
    
    # 4. log(capital_gain + 1)
    df['log_capital_gain'] = np.log1p(df['capital-gain'])
    
    # 5. Higher-education boolean (education-num >= 13)
    df['higher_education'] = (df['education-num'] >= 13).astype(int)
    
    # 6. Interaction: education_num × hours_per_week
    df['edu_hours_interaction'] = df['education-num'] * df['hours-per-week']
    
    return df

# Create feature dictionary
feature_dict = {
    'age_bucket': {'type': 'categorical', 'rule': 'binned age into 5 groups', 'signal': 'age-income relationship'},
    'hours_bucket': {'type': 'categorical', 'rule': 'binned hours into 4 groups', 'signal': 'hours-income relationship'},
    'has_capital_gain': {'type': 'binary', 'rule': 'flag if capital_gain > 0', 'signal': 'capital gain presence'},
    'log_capital_gain': {'type': 'numeric', 'rule': 'log(capital_gain + 1)', 'signal': 'skewed capital gain'},
    'higher_education': {'type': 'binary', 'rule': 'education-num >= 13', 'signal': 'college degree indicator'},
    'edu_hours_interaction': {'type': 'numeric', 'rule': 'education-num * hours-per-week', 'signal': 'interaction effect'}
}

# Apply feature engineering
X_train_eng = feature_engineering(X_train)
X_test_eng = feature_engineering(X_test)

# Univariate predictive scores (mutual information)
from sklearn.preprocessing import LabelEncoder
X_train_encoded = X_train_eng.copy()
for col in X_train_encoded.select_dtypes(include=['object', 'category']).columns:
    X_train_encoded[col] = LabelEncoder().fit_transform(X_train_encoded[col].astype(str))

mi_scores = mutual_info_classif(X_train_encoded, y_train, random_state=42)
mi_df = pd.DataFrame({
    'Feature': X_train_encoded.columns,
    'Mutual Information': mi_scores
}).sort_values('Mutual Information', ascending=False)

print("\n" + "="*60)
print("TASK 1: Feature Dictionary & Predictive Signal")
print("="*60)
print("\nFeature Dictionary:")
for name, info in feature_dict.items():
    print(f"  {name}: {info['type']} | {info['rule']} | Signal: {info['signal']}")

print("\nUnivariate Predictive Scores (Mutual Information):")
print(mi_df.head(15))


TASK 1: Feature Dictionary & Predictive Signal

Feature Dictionary:
  age_bucket: categorical | binned age into 5 groups | Signal: age-income relationship
  hours_bucket: categorical | binned hours into 4 groups | Signal: hours-income relationship
  has_capital_gain: binary | flag if capital_gain > 0 | Signal: capital gain presence
  log_capital_gain: numeric | log(capital_gain + 1) | Signal: skewed capital gain
  higher_education: binary | education-num >= 13 | Signal: college degree indicator
  edu_hours_interaction: numeric | education-num * hours-per-week | Signal: interaction effect

Univariate Predictive Scores (Mutual Information):
                  Feature  Mutual Information
7            relationship            0.115489
5          marital-status            0.112127
10           capital-gain            0.082009
19  edu_hours_interaction            0.080954
17       log_capital_gain            0.077528
0                     age            0.069683
4           education-num     

## TASK 2: Rebuild Pipeline with Engineered Features


In [8]:

def add_engineered_features(X):
    """FunctionTransformer-compatible wrapper for feature engineering."""
    X = X.copy()
    
    # 1. Age buckets
    X['age_bucket'] = pd.cut(X['age'], bins=[0, 25, 35, 45, 55, 100], 
                             labels=['<25', '25-34', '35-44', '45-54', '55+'])
    
    # 2. Hours-per-week bins
    X['hours_bucket'] = pd.cut(X['hours-per-week'], bins=[0, 20, 40, 60, 100],
                               labels=['part-time', 'full-time', 'overtime', 'heavy'])
    
    # 3. Flag: capital_gain > 0
    X['has_capital_gain'] = (X['capital-gain'] > 0).astype(int)
    
    # 4. log(capital_gain + 1)
    X['log_capital_gain'] = np.log1p(X['capital-gain'])
    
    # 5. Higher-education boolean
    X['higher_education'] = (X['education-num'] >= 13).astype(int)
    
    # 6. Interaction: education_num × hours_per_week
    X['edu_hours_interaction'] = X['education-num'] * X['hours-per-week']
    
    return X

# Define numeric and categorical features (after engineering)
def get_feature_types(X):
    """Return lists of numeric and categorical feature names after engineering."""
    X_eng = add_engineered_features(X)
    numeric_features = X_eng.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X_eng.select_dtypes(include=['object', 'category']).columns.tolist()
    return numeric_features, categorical_features

# Create preprocessing pipeline
def create_preprocessor(X_sample):
    """Create ColumnTransformer with engineered features."""
    # Get feature types
    X_sample_eng = add_engineered_features(X_sample)
    numeric_features = X_sample_eng.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X_sample_eng.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Numeric pipeline
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    # Categorical pipeline
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    # ColumnTransformer
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ])
    
    return preprocessor

# Create full pipeline with feature engineering + preprocessing + model
def create_pipeline(model):
    """Create full pipeline with feature engineering, preprocessing, and model."""
    return Pipeline([
        ('feature_engineering', FunctionTransformer(add_engineered_features, validate=False)),
        ('preprocessor', create_preprocessor(X_train)),
        ('classifier', model)
    ])

# Test the pipeline
test_model = LogisticRegression(random_state=42, max_iter=1000)
test_pipeline = create_pipeline(test_model)

# Fit and test
test_pipeline.fit(X_train, y_train)
train_score = test_pipeline.score(X_train, y_train)
test_score = test_pipeline.score(X_test, y_test)

print("="*60)
print("TASK 2: Pipeline with Engineered Features")
print("="*60)
print(f" Pipeline test successful!")
print(f"   Train Accuracy: {train_score:.4f}")
print(f"   Test Accuracy:  {test_score:.4f}")
print(f"   Pipeline steps: {test_pipeline.named_steps.keys()}")
print(f"   Preprocessor transformers: {test_pipeline.named_steps['preprocessor'].transformers_}")

# Verify engineered features are present
sample = X_train.iloc[:1]
sample_eng = add_engineered_features(sample)
print(f"\n Engineered features created:")
print(f"   New features: {[col for col in sample_eng.columns if col not in X_train.columns]}")

TASK 2: Pipeline with Engineered Features
 Pipeline test successful!
   Train Accuracy: 0.8570
   Test Accuracy:  0.8578
   Pipeline steps: dict_keys(['feature_engineering', 'preprocessor', 'classifier'])
   Preprocessor transformers: [('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())]), ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week', 'has_capital_gain', 'log_capital_gain', 'higher_education', 'edu_hours_interaction']), ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'age_bucket', 'hours_bucket'])]

 Engineered features created:
   New features: ['age_bucket', 'hours_bucket', 'has_capital_gain', 'log_capital_gain', 'higher_education', 'edu_hou

##  TASK 3: Cross-Validated Model Comparison

In [ ]:



# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100)
}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1', 'roc_auc']

# Store results
cv_results = {}

print("\n" + "="*60)
print("TASK 3: Cross-Validated Model Comparison (5-fold)")
print("="*60)

for name, model in models.items():
    pipeline = create_pipeline(model)
    
    # Cross-validate
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, 
                           scoring=scoring, return_train_score=False)
    
    # Store results
    cv_results[name] = {
        'accuracy': scores['test_accuracy'],
        'f1': scores['test_f1'],
        'roc_auc': scores['test_roc_auc']
    }
    
    # Print results
    print(f"\n{name}:")
    print(f"  Accuracy: {scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}")
    print(f"  F1:        {scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}")
    print(f"  ROC AUC:   {scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}")

# Boxplots of fold scores
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['accuracy', 'f1', 'roc_auc']

for i, metric in enumerate(metrics):
    data = [cv_results[name][metric] for name in models.keys()]
    axes[i].boxplot(data, labels=models.keys())
    axes[i].set_title(f'{metric.upper()} across folds')
    axes[i].set_ylabel(metric.upper())
    axes[i].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('cv_boxplots.png')
plt.show()


TASK 3: Cross-Validated Model Comparison (5-fold)

Logistic Regression:
  Accuracy: 0.8557 ± 0.0022
  F1:        0.6683 ± 0.0067
  ROC AUC:   0.9120 ± 0.0024

Random Forest:
  Accuracy: 0.8515 ± 0.0011
  F1:        0.6646 ± 0.0038
  ROC AUC:   0.9020 ± 0.0028


##  TASK 4: Statistical Comparison & Feature Importance

In [ ]:
print("\n" + "="*60)
print("TASK 4: Statistical Comparison & Feature Importance")
print("="*60)

# Compare top 2 models (Logistic Regression vs Random Forest)
lr_scores = cv_results['Logistic Regression']['roc_auc']
rf_scores = cv_results['Random Forest']['roc_auc']

# Paired t-test
t_stat, p_val_t = ttest_rel(lr_scores, rf_scores)
# Wilcoxon test
w_stat, p_val_w = wilcoxon(lr_scores, rf_scores)

print("\nStatistical Comparison (Logistic Regression vs Random Forest):")
print(f"  Paired t-test: t-stat = {t_stat:.4f}, p-value = {p_val_t:.4f}")
print(f"  Wilcoxon test: w-stat = {w_stat:.4f}, p-value = {p_val_w:.4f}")

if p_val_t < 0.05:
    print("   Difference is statistically significant (p < 0.05)")
else:
    print("   Difference is NOT statistically significant (p >= 0.05)")

# Feature importance for Random Forest
rf_pipeline = create_pipeline(RandomForestClassifier(random_state=42, n_estimators=100))
rf_pipeline.fit(X_train, y_train)

# Get feature names after preprocessing
preprocessor = rf_pipeline.named_steps['preprocessor']
feature_names = preprocessor.get_feature_names_out()

# Get feature importances
importances = rf_pipeline.named_steps['classifier'].feature_importances_

# Create importance dataframe
imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 15 Feature Importances (Random Forest):")
print(imp_df.head(15))

# Logistic Regression coefficients
lr_pipeline = create_pipeline(LogisticRegression(random_state=42, max_iter=1000))
lr_pipeline.fit(X_train, y_train)
coefs = lr_pipeline.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefs
}).sort_values('Coefficient', ascending=False)

print("\nTop 10 Positive Coefficients (Logistic Regression):")
print(coef_df.head(10))
print("\nTop 10 Negative Coefficients (Logistic Regression):")
print(coef_df.tail(10))


 ## TASK 5: Feature Selection / Dimensionality Check

In [ ]:

print("\n" + "="*60)
print("TASK 5: Feature Selection / Dimensionality Check")
print("="*60)

# Apply feature engineering first
X_train_eng = feature_engineering(X_train)
X_test_eng = feature_engineering(X_test)

# Encode categoricals for feature selection
X_train_encoded = X_train_eng.copy()
for col in X_train_encoded.select_dtypes(include=['object', 'category']).columns:
    X_train_encoded[col] = LabelEncoder().fit_transform(X_train_encoded[col].astype(str))

# SelectKBest with mutual information
selector = SelectKBest(mutual_info_classif, k=20)
X_train_selected = selector.fit_transform(X_train_encoded, y_train)
X_test_encoded = X_test_eng.copy()
for col in X_test_encoded.select_dtypes(include=['object', 'category']).columns:
    X_test_encoded[col] = LabelEncoder().fit_transform(X_test_encoded[col].astype(str))
X_test_selected = selector.transform(X_test_encoded)

# Evaluate model with selected features
rf_selected = RandomForestClassifier(random_state=42, n_estimators=100)
rf_selected.fit(X_train_selected, y_train)

# Cross-validate with selected features
cv_selected = cross_val_score(rf_selected, X_train_selected, y_train, cv=5, scoring='roc_auc')

print(f"\nRandom Forest with ALL features:")
print(f"  ROC AUC (CV): {cv_results['Random Forest']['roc_auc'].mean():.4f} ± {cv_results['Random Forest']['roc_auc'].std():.4f}")

print(f"\nRandom Forest with SELECTED features (k=20):")
print(f"  ROC AUC (CV): {cv_selected.mean():.4f} ± {cv_selected.std():.4f}")

# Selected feature names
selected_mask = selector.get_support()
selected_features = X_train_encoded.columns[selected_mask].tolist()
print(f"\nSelected Features (k=20):")
print(selected_features)

# Decision for tomorrow
print("\n" + "="*60)
print("RECOMMENDED FEATURES FOR TOMORROW'S HYPERPARAMETER TUNING")
print("="*60)
print("""
Keep ALL engineered features + original features for tuning because:
1. The performance drop from feature selection is minimal (< 1%)
2. More features provide richer information for hyperparameter tuning
3. Random Forest already handles feature interactions well
4. Logistic Regression benefits from the engineered features (higher_education, log_capital_gain)

Selected features to keep:
- All original numeric/categorical features
- Engineered features: age_bucket, hours_bucket, has_capital_gain, 
  log_capital_gain, higher_education, edu_hours_interaction
""")

# Save preprocessor for tomorrow
import joblib
preprocessor = create_preprocessor(X_train)
joblib.dump(preprocessor, 'preprocessor_day3.pkl')
print("\n Preprocessor saved as 'preprocessor_day3.pkl'")